### Genollama: Training data generation

Set up environment (including `granted --env` credentials if required)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datagen import BedrockBatchGenerator, TrainingDataUploader
from genoschema.prompt_builder import PromptBuilder
from genoschema.schema import GenomicTestReport

In [ ]:
pb = PromptBuilder()

#### Download documents from batch

In [ ]:
gen = BedrockBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    user_prompt_function=lambda doc: doc['content'],
    schema=GenomicTestReport,
    schema_name="genoschema",
    model_name="sonnet4",
    document_batches=["genomics-batch-2026-01-25-001.tar.gz"],
)

#### Start batch generation

In [ ]:
gen.generate_via_batch(1000, os.environ["BEDROCK_EXECUTION_ROLE"], os.environ["BUCKET"])

#### Download and parse batch outputs

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=GenomicTestReport,
    schema_name="genoschema",
    system_prompt=pb.build_main_prompt(),
    short_description="genollama-batch",
    long_description="Training data for GenoLlama",
    input_folder=Path("./data/trainingdata"),
)